# PhysLLaVA Pipeline Progress Tracking

**Date:** 2026-03-13  
**Cluster:** Harvard FASRC  
**Hardware:** NVIDIA A100-SXM4-80GB  

This notebook tracks the full PhysLLaVA pipeline execution: data preparation, tokenization,
two-stage training, and evaluation.

---

## Phase 0: Environment Setup
**Status:** ✅ Complete  
**Timestamp:** 2026-03-12 22:00

### Actions Taken
- Created `physllava` conda environment (Python 3.11)
- Installed PyTorch 2.5.1+cu121 (CUDA 12.1 backend)
- Installed all project requirements from `requirements.txt` (skipped `vllm`)
- Flash-attn: **SKIPPED** (build failed on cluster CUDA configuration)
- GPU confirmed: NVIDIA A100-SXM4-80GB (80GB VRAM)
- HF_TOKEN confirmed valid (Llama 3.1 8B accessible)

### Config Changes
- `llm.use_flash_attention: false` (flash-attn unavailable)
- `logging.use_wandb: false` (WANDB_API_KEY not set)
- `dataset.num_jets_per_class: 500` (reduced from 3000 for speed)

## Phase 1a: Data Download
**Status:** ✅ Complete (with fallback)  
**Timestamp:** 2026-03-12 22:30

### Decision: Synthetic Data Fallback
- JetClass-II HuggingFace streaming download was extremely slow
  - The dataset is stored in mixed-class format requiring scanning millions of records
  - Process used 93% CPU and 2.3GB RAM for 35+ minutes without completing
- **Solution**: Implemented `data/generate_synthetic_jets.py`
  - Generates physics-motivated jet distributions for each class
  - Uses known LHC jet kinematic distributions (pT, eta, mass, N-subjettiness)
  - Generates realistic constituent kinematics with Gaussian angular smearing

### Output
- 500 jets per class × 10 classes = **5,000 total jets**
- Saved as parquet files in `jetclass2_subset/`

## Phase 1b: OmniJet-alpha Setup & Jet Tokenization
**Status:** ✅ Complete (with fallback)  
**Timestamp:** 2026-03-12 23:00

### OmniJet-alpha Status
- Cloned repo: `/hep-llava-data/omnijet_alpha/`
- Downloaded checkpoint: `checkpoints/vqvae_8192_tokens/model_ckpt.ckpt`
- Installed dependencies: `vqtorch` (from GitHub), `lightning`, `awkward`, `vector`
- Fixed `pkg_resources` issue: downgraded setuptools to <81
- **Import verified**: `VQVAELightning` imports successfully

### Decision: Simple Discretization Fallback
- OmniJet-alpha VQ-VAE expects raw pT values in GeV with specific preprocessing
- Synthetic data uses pt_rel (normalized), requiring inverse transformation
- **Solution**: Simple 3D binning discretization (32^3 = 32768 codes)
  - Bins: pt_rel ∈ [0,1], eta_rel ∈ [-0.8,0.8], phi_rel ∈ [-0.8,0.8]
  - Index = pt_bin × 32² + eta_bin × 32 + phi_bin

### Config Changes
- `tokenizer.codebook_size: 32768`
- `physics_encoder.vocab_size: 32768`

### Output
- `tokenized_jets/tokenized_jets.json` (5000 jets with metadata)
- `tokenized_jets/token_indices.npy` (5000 × 128 int64 array)
- `tokenized_jets/masks.npy` (5000 × 128 bool array)

## Phase 1c: Caption & QA Generation
**Status:** ✅ Complete  
**Timestamp:** 2026-03-12 23:05

### Caption Generation (Strategy 1+3)
- 2 rule-based + 3 slot-fill captions per jet = 5 captions × 5000 jets
- Total: **25,000 caption conversations**
- LLM captions skipped (OPENROUTER_KEY not set)

### QA Generation
- 4 factual + 4 kinematic + 4 reasoning QA pairs per jet
- Total: **60,000 QA pairs**
  - Factual: 20,000 (process ID, constituent count, b-quark presence, etc.)
  - Kinematic: 20,000 (pT, mass, eta, energy, tau21)
  - Reasoning: 20,000 (origin reasoning, topology description, tagging features)

## Phase 2: Stage 1 Training (Feature Alignment)
**Status:** ✅ Complete  
**Timestamp:** 2026-03-12 23:21 → 00:00

### Architecture
- **Frozen**: Llama 3.1 8B (8.03B params)
- **Trainable**: PhysicsEncoder (35.8M) + MLPProjector (18.9M) = 54.6M params (0.68%)

### Training Config
| Parameter | Value |
|-----------|-------|
| Batch size | 8 |
| Max text length | 256 |
| Learning rate | 1e-3 |
| Epochs | 1 |
| Total steps | 3,125 |
| Gradient checkpointing | Enabled |

### Training Results
| Metric | Value |
|--------|-------|
| Initial loss | 14.64 |
| Final loss | 0.0875 |
| Epoch average loss | 0.1169 |
| Training time | 36.5 minutes |
| Throughput | ~1.43 it/s |

### Troubleshooting
- First run OOM'd with batch_size=64 (77.88GB used out of 79.25GB)
- Fixed by: batch_size=8 + gradient checkpointing + max_text_length=256

## Phase 3: Stage 2 Training (Instruction Tuning)
**Status:** ✅ Complete  
**Timestamp:** 2026-03-13 01:05 → 01:45

### Architecture
- **LoRA on LLM**: r=32, alpha=64, dropout=0.05 (all 7 weight matrices)
- **Trainable**: PhysicsEncoder + MLPProjector + LoRA = 138.5M params (1.70%)

### Dataset
- Stage 2 uses subset: 5K captions + 10K QA = **15,000 samples** (from 85K total)
- Reduced to keep training time under 1 hour

### Training Config
| Parameter | Value |
|-----------|-------|
| Batch size | 4 |
| Grad accumulation | 4 |
| Effective batch | 16 |
| Learning rate | 2e-5 |
| Epochs | 1 |
| Total steps | 3,750 |

### Training Results
| Metric | Value |
|--------|-------|
| Final loss | 0.0230 |
| Epoch average loss | 0.0506 |
| Training time | 39 minutes |
| Throughput | ~1.6 it/s |

### Loaded Stage 1 Checkpoint
- Physics encoder and projector weights initialized from Stage 1 final checkpoint
- LLM initialized from base Llama 3.1 8B weights with fresh LoRA adapters

## Phase 4: Evaluation
**Status:** ✅ Complete  
**Timestamp:** 2026-03-13 01:50 → 02:10

### Evaluation Setup
- 50 jets per class × 10 classes = 500 jets
- Process identification: keyword matching on generated responses
- Kinematic QA: first 100 jets per question type

In [ ]:
import json
from pathlib import Path

data_dir = Path('/n/holystore01/LABS/iaifi_lab/Users/sambt/hep-llava-data')
eval_dir = data_dir / 'eval_results'

# Load results
with open(eval_dir / 'eval_results.json') as f:
    results = json.load(f)

print('=== Process Identification ===')
proc = results['process_identification']
print(f"Accuracy: {proc['accuracy']:.4f} ({proc['accuracy']*100:.1f}%)")
print(f"Valid predictions: {proc['valid_predictions']}/{proc['total_samples']}")
print()
print('=== Kinematic QA ===')
for q, metrics in results['kinematic_qa'].items():
    mre = metrics['mean_relative_error']
    print(f"{q}: mean relative error = {mre:.4f} ({mre*100:.1f}%)")

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

# Load responses
with open(eval_dir / 'process_id_responses.json') as f:
    responses = json.load(f)

classes = ['Hbb', 'Hcc', 'Hgg', 'H4q', 'Hqql', 'Zqq', 'Wqq', 'Tbqq', 'Tbl', 'QCD']

# Per-class accuracy
class_acc = {}
for cls in classes:
    cls_r = [r for r in responses if r['true'] == cls]
    correct = sum(1 for r in cls_r if r.get('predicted') == cls)
    class_acc[cls] = correct / len(cls_r) if cls_r else 0

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(classes, [class_acc[c] for c in classes], color='steelblue', edgecolor='black')
ax.axhline(0.1, color='red', linestyle='--', label='Random (10%)', linewidth=2)
ax.set_xlabel('Jet Class', fontsize=12)
ax.set_ylabel('Classification Accuracy', fontsize=12)
ax.set_title('PhysLLaVA Per-Class Process Identification Accuracy\n(1 epoch training, 500 eval jets)', fontsize=13)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=12)

# Add value labels
for bar, cls in zip(bars, classes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{class_acc[cls]:.0%}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import seaborn as sns

# Confusion matrix
conf_matrix = np.zeros((len(classes), len(classes)), dtype=int)
for r in responses:
    true_cls = r['true']
    pred_cls = r.get('predicted', None)
    if pred_cls in classes:
        i = classes.index(true_cls)
        j = classes.index(pred_cls)
        conf_matrix[i, j] += 1

# Normalize by row
conf_norm = conf_matrix.astype(float)
row_sums = conf_norm.sum(axis=1, keepdims=True)
conf_norm = conf_norm / (row_sums + 1e-8)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(conf_norm, xticklabels=classes, yticklabels=classes,
            annot=True, fmt='.2f', cmap='Blues', ax=ax, vmin=0, vmax=1)
ax.set_xlabel('Predicted Class', fontsize=13)
ax.set_ylabel('True Class', fontsize=13)
ax.set_title('Normalized Confusion Matrix - Process Identification', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Show some example model responses
print('=== Example Model Responses ===')
shown = {'correct': 0, 'wrong': 0}
for r in responses:
    is_correct = r.get('predicted') == r['true']
    key = 'correct' if is_correct else 'wrong'
    if shown[key] < 2:
        print(f"--- {'CORRECT' if is_correct else 'WRONG'} ---")
        print(f"True: {r['true']} | Predicted: {r.get('predicted', 'unknown')}")
        print(f"Response: {r['response'][:300]}")
        print()
        shown[key] += 1
    if shown['correct'] >= 2 and shown['wrong'] >= 2:
        break

## Phase 5: Results Summary

### Key Findings

1. **The pipeline works end-to-end**: From jet tokenization through LLM generation, all components connect correctly.

2. **Above-chance performance after 1 epoch**: 15.4% vs 10% random baseline
   - Best class: Hcc (64%!) - the model strongly associates charm quarks with Higgs
   - Good: H4q (28%), Hqql (38%)
   - Missed: Hbb, Hgg, Zqq, Wqq, Tbl, Zqq (similar to other classes)

3. **Constituent count regression is excellent**: 3% mean relative error
   - The model learned to read constituent count from the physics tokens
   - This suggests the tokenization scheme carries constituent information

4. **Mass regression is reasonable**: 13.2% error
   - Higgs mass (~125 GeV), W (~80 GeV), Z (~91 GeV) are distinct but close

5. **Model generates fluent physics text**: Responses correctly use particle physics
   terminology (N-subjettiness, soft-drop mass, boosted topologies)

### Limitations
- Synthetic data lacks the full statistical diversity of real collisions
- Simple tokenizer doesn't capture the learned structure of OmniJet VQ-VAE
- 1 epoch is insufficient for strong performance; 5+ epochs would be better
- The model has a prediction bias toward Hcc (268/500 predictions)

### Next Steps
1. Download real JetClass-II data (non-streaming, class-specific splits)
2. Integrate OmniJet-alpha VQ-VAE tokenizer properly
3. Train for 5 epochs each stage
4. Add class-balanced sampling to avoid prediction collapse

---

## Phase 6: Real JetClass-II Pipeline Run (2026-03-13)

**Status:** Complete  
**This section documents the re-run with real JetClass-II data and OmniJet VQ-VAE tokenization.**

### Data Pipeline (Pre-verified)
- **Real JetClass-II data**: 10 classes × 3000 jets = 30,000 total jets
  - Classes: Res2P_bb, Res2P_cc, Res2P_ss, Res2P_uu, Res2P_gg, Res2P_WW4q, Res2P_WWlv, Res2P_ZZ4q, QCD_187, QCD_185
- **OmniJet-alpha VQ-VAE tokenization** (8192-code codebook):
  - `token_indices.npy`: shape (30000, 128), range 0–8188
  - `masks.npy`: shape (30000, 128), avg 46.7 valid particles/jet
- **Captions**: 150,000 conversations (rule-based + slot-fill, 5 per jet)
- **QA data**: 360,000 pairs (15 per jet × 3 difficulty levels)

### Stage 1 Training (Feature Alignment)

| Parameter | Value |
|-----------|-------|
| Batch size | 32 (reduced from 64 — OOM) |
| Learning rate | 1e-3 |
| Epochs | 1 (of 5 in config; process killed after epoch 1) |
| Steps | 4687 (150K / 32) |
| Trainable params | 42.1M (0.52%) |
| Runtime | ~3.5 hours on A100 80GB |

**Loss curve:**

| Step | Loss |
|------|------|
| 0 | 14.39 |
| 100 | 0.172 |
| 500 | 0.097 |
| 1000 | 0.093 |
| 2000 | 0.098 |
| 4000 | 0.080 |
| 4687 | 0.072 |
| **Epoch 1 avg** | **0.1032** |

### Stage 2 Training (Instruction Tuning with LoRA)

| Parameter | Value |
|-----------|-------|
| Batch size | 16 |
| Gradient accumulation | 4 (effective batch 64) |
| Learning rate | 2e-5 |
| Epochs | 1 |
| Training data | 90K subset (30K captions + 60K QA) |
| LoRA rank/alpha | 32/64 |
| LoRA trainable | 83.9M (1.03% of LLM) |
| Total trainable | 125.9M (1.54%) |
| Runtime | ~3.5 hours on A100 80GB |

**Loss curve:**

| Batch | Loss |
|-------|------|
| 0 | 0.240 |
| 100 | 0.161 |
| 500 | 0.056 |
| 1000 | 0.056 |
| 3000 | 0.044 |
| 5625 | 0.041 |
| **Epoch avg** | **0.0462** |

### Evaluation Results (2000 jets, 200/class)

| Metric | Value |
|--------|-------|
| Process ID Accuracy | **9.65%** (random: 10%) |
| Constituent Count Error | **0.70%** (median: 0%) |
| pT Mean Relative Error | 42.7% (median: 23.9%) |
| Mass Mean Relative Error | 139.6% (median: 45.2%) |

**Per-class accuracy (highlights):**
- Res2P_WWlv: 58% recall (best)
- Res2P_WW4q: 38% recall
- All other classes: 0–0.5% recall (prediction collapse to WWlv/WW4q)

**Root cause of low accuracy:**
1. Many caption templates for Res2P_bb/ss/uu/gg are non-discriminative ("This is a heavy resonance X jet")
2. 1 training epoch insufficient to learn VQ-token → class discrimination
3. Mode collapse: model predicts Res2P_WWlv (57.5%) and Res2P_WW4q (41.5%) for nearly all inputs

In [ ]:
import json
import numpy as np
from pathlib import Path
from collections import Counter

# Load the new evaluation results (real JetClass-II data)
data_dir = Path('/n/holystore01/LABS/iaifi_lab/Users/sambt/hep-llava-data')
with open(data_dir / 'eval_results/eval_results.json') as f:
    results = json.load(f)

with open(data_dir / 'eval_results/process_id_responses.json') as f:
    responses = json.load(f)

# Real JetClass-II classes
classes = ['Res2P_bb', 'Res2P_cc', 'Res2P_ss', 'Res2P_uu', 'Res2P_gg',
           'Res2P_WW4q', 'Res2P_WWlv', 'Res2P_ZZ4q', 'QCD_187', 'QCD_185']

# Per-class accuracy
class_acc = {}
for cls in classes:
    cls_r = [r for r in responses if r['true'] == cls]
    correct = sum(1 for r in cls_r if r.get('predicted') == cls)
    class_acc[cls] = correct / len(cls_r) if cls_r else 0

print('=== Real JetClass-II Evaluation Results ===')
print(f"Overall accuracy: {results['process_identification']['accuracy']:.4f} ({results['process_identification']['accuracy']*100:.1f}%)")
print(f"Random baseline: 10.0%")
print()
print('Per-class recall:')
for cls in classes:
    acc = class_acc[cls]
    bar = '#' * int(acc * 30)
    print(f"  {cls:<18} {acc:.3f} ({acc*100:.1f}%) {bar}")

print()
print('Prediction distribution:')
preds = Counter(r['predicted'] for r in responses)
for cls, cnt in preds.most_common():
    print(f"  {cls}: {cnt} ({100*cnt/len(responses):.1f}%)")

print()
print('Kinematic regression:')
for q, m in results['kinematic_qa'].items():
    mre = m['mean_relative_error']
    med = m['median_relative_error']
    print(f"  {q}: mean_rel_err={mre:.3f} ({mre*100:.1f}%), median_rel_err={med:.3f} ({med*100:.1f}%)")